# Phase 1: Dataset Exploration and Cleaning
**Project:** AI Mental Health Support Chatbot  
**Dataset URL:** [Sentiment Analysis for Mental Health (Kaggle)](https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health)  
**Goal:** Explore, analyze, and conservatively clean the dataset for downstream fine-tuning of DistilBERT.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path to enable src imports
sys.path.append("..")
from src.data.cleaning import load_raw_data, inspect_raw_data, clean_dataset, save_processed_data, run_cleaning_pipeline

## 1. Load Raw Dataset
We load the untouched dataset from `data/raw/Combined Data.csv`.

In [ ]:
raw_path = "../data/raw/Combined Data.csv"
df_raw = load_raw_data(raw_path)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

## 2. Dataset Structure & Data Types
Inspect column names, data types, and non-null counts.

In [ ]:
df_raw.info()

## 3. Missing Value Analysis
Check for missing (`NaN`) values across all columns.

In [ ]:
null_series = df_raw.isnull().sum()
print("Missing value count per column:")
print(null_series)

## 4. Duplicate & Conflicting Label Analysis
- **Exact Full Row Duplicates:** Identical `statement` and `status` values.
- **Duplicate Statements:** Multiple occurrences of the same text.
- **Conflicting Labels:** Statements assigned to multiple different target categories.

In [ ]:
raw_stats = inspect_raw_data(df_raw)
print(f"Full row duplicates: {raw_stats['full_row_duplicates']}")
print(f"Duplicate statements (total): {raw_stats['duplicate_statements']}")
print(f"Conflicting statements: {raw_stats['conflicting_statement_count']} unique texts ({raw_stats['conflicting_rows_count']} rows)")

# Inspect sample conflicting statements
if raw_stats['conflicting_texts']:
    print("\nSample conflicting statement:")
    sample_text = raw_stats['conflicting_texts'][0]
    print(repr(sample_text))
    print("Associated labels:", df_raw[df_raw['statement'] == sample_text]['status'].tolist())

## 5. Text Length & Short Text Analysis
We analyze character lengths and word counts to check for extremely short or malformed entries.

In [ ]:
valid_text = df_raw.dropna(subset=['statement']).copy()
valid_text['char_len'] = valid_text['statement'].str.len()
valid_text['word_count'] = valid_text['statement'].str.split().str.len()

print("Character and Word Count Statistics:")
print(valid_text[['char_len', 'word_count']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\nVery short texts (word count <= 2):", (valid_text['word_count'] <= 2).sum())
valid_text[valid_text['word_count'] <= 2][['statement', 'status']].head(10)

## 6. Raw Label Distribution & Class Imbalance
Analyze target label counts and percentage breakdown.

In [ ]:
label_counts = df_raw['status'].value_counts()
label_pcts = (label_counts / len(df_raw)) * 100
dist_df = pd.DataFrame({'Count': label_counts, 'Percentage (%)': label_pcts.round(2)})
print(dist_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=label_counts.values, y=label_counts.index, palette="viridis")
plt.title("Raw Label Distribution")
plt.xlabel("Count")
plt.ylabel("Mental Health Status")
plt.tight_layout()
plt.show()

## 7. Conservative Text Cleaning
We execute our conservative cleaning decisions:
1. Drop index column (`Unnamed: 0`).
2. Strip whitespace from `statement` and `status`.
3. Remove 362 null/empty text statements.
4. Remove 49 rows belonging to 18 conflicting statements.
5. Remove exact duplicate statements, keeping the first occurrence.

**Linguistic Context Preservation:** No stopwords, emojis, or punctuation are removed, and no stemming or lemmatization is applied.

In [ ]:
df_clean = clean_dataset(df_raw)
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Data retained: {(len(df_clean) / len(df_raw)) * 100:.2f}%")

## 8. Before vs After Comparison & Export
Compare class distributions before and after cleaning, and save the processed CSV.

In [ ]:
before_counts = df_raw['status'].value_counts()
after_counts = df_clean['status'].value_counts()
comp_df = pd.DataFrame({
    "Raw Count": before_counts,
    "Cleaned Count": after_counts,
    "Removed": before_counts - after_counts,
    "% Retained": ((after_counts / before_counts) * 100).round(2)
})
print(comp_df)

save_processed_data(df_clean, "../data/processed/cleaned_mental_health_data.csv")